In [1]:
import warnings
warnings.filterwarnings(action="ignore")

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch
print("CUDA Available:", torch.cuda.is_available())
print("GPU Device Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA Available: True
GPU Device Name: NVIDIA GeForce RTX 5080


Cell below is used to speed up sklearn with gpu, will not work unless using Nvidia Rapids JuptyerLab

In [4]:
%load_ext cuml.accel

In [5]:
data = pd.read_csv('./sleep_health_dataset.csv')
data.head(10)

,person_id,age,gender,occupation,bmi,country,sleep_duration_hrs,sleep_quality_score,rem_percentage,deep_sleep_percentage,...,heart_rate_resting_bpm,sleep_aid_used,shift_work,room_temperature_celsius,weekend_sleep_diff_hrs,season,day_type,cognitive_performance_score,sleep_disorder_risk,felt_rested
0,1,29,Female,Driver,25.7,Japan,6.19,6.6,22.5,19.3,...,63,0,0,20.1,1.84,Autumn,Weekday,73.4,Healthy,0
1,2,55,Female,Software Engineer,22.0,USA,8.32,6.9,26.9,14.9,...,52,1,0,18.0,0.13,Winter,Weekend,99.4,Healthy,1
2,3,42,Male,Nurse,25.0,India,3.74,1.0,20.2,16.2,...,72,0,1,17.9,1.67,Spring,Weekend,2.5,Severe,0
3,4,37,Female,Student,29.5,India,6.79,6.4,17.7,17.7,...,71,0,0,19.1,2.37,Summer,Weekend,67.8,Healthy,0
4,5,23,Male,Lawyer,23.6,Spain,5.02,3.2,23.3,18.3,...,71,0,0,19.7,1.26,Summer,Weekday,38.1,Mild,0
5,6,23,Female,Driver,25.5,Brazil,8.16,5.7,17.3,20.1,...,71,1,0,22.7,0.70,Summer,Weekend,49.9,Mild,1
6,7,20,Female,Software Engineer,18.2,Netherlands,7.27,5.1,17.5,18.8,...,79,0,0,17.6,0.73,Spring,Weekend,57.9,Mild,1
7,8,48,Female,Freelancer,31.5,UK,8.04,6.5,25.3,22.3,...,60,1,0,21.6,0.47,Winter,Weekend,92.0,Mild,0
8,9,37,Male,Manager,29.9,USA,6.04,5.4,18.9,15.7,...,74,0,0,17.8,1.80,Autumn,Weekend,53.6,Mild,0
9,10,41,Female,Nurse,34.8,USA,6.22,4.9,21.4,15.5,...,80,1,0,24.4,1.51,Autumn,Weekday,67.8,Mild,1


In [6]:
!pip install category_encoders

In [7]:
# drop irrelavent columns

data = data.dropna()
data = data[data['gender'] != "Other"]

regressionTarget = data[['cognitive_performance_score']]

data = data.drop(['person_id', 'cognitive_performance_score', 'sleep_disorder_risk', 'shift_work', 'felt_rested'], axis=1)

In [8]:
from category_encoders.binary import BinaryEncoder
#Dataset Preprocessing
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
import category_encoders as ce

num_pipeline = Pipeline([
        ('std_scaler', StandardScaler()),
    ])

binary_pipeline = Pipeline([
        ("binary_encoder", ce.BinaryEncoder())
    ])

num_attribs = ['age', 'bmi', 'sleep_duration_hrs', 'sleep_quality_score', 'rem_percentage', 'deep_sleep_percentage', 'sleep_latency_mins',
       'wake_episodes_per_night', 'caffeine_mg_before_bed', 'alcohol_units_before_bed', 'screen_time_before_bed_mins',
       'exercise_day', 'steps_that_day', 'nap_duration_mins', 'stress_score',
       'work_hours_that_day', 'heart_rate_resting_bpm', 'room_temperature_celsius', 'weekend_sleep_diff_hrs']

binary_attribs = cols=['country', 'occupation', 'chronotype', 'mental_health_condition', 'season', 'day_type']

full_pipeline = ColumnTransformer([
        ("num", num_pipeline, num_attribs),
        ("binary", binary_pipeline, binary_attribs)
    ]).fit(data)

In [10]:
#Dataset Split
from sklearn.model_selection import train_test_split

xTrainR, xTestR, yTrainR, yTestR = train_test_split(data, regressionTarget, test_size=0.33, random_state=42)
xTrainR = full_pipeline.transform(xTrainR)
xTestR = full_pipeline.transform(xTestR)

yTrainR = yTrainR.to_numpy()
yTestR = yTestR.to_numpy()

print(f'Training Set Size: {xTrainR.shape}\nTest Set Size: {xTestR.shape}')

Training Set Size: (65677, 37)
Test Set Size: (32349, 37)


In [11]:
!pip install xgboost

In [12]:
#Regression
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import mean_absolute_error
import xgboost as xgb

In [13]:
print("Training Linear Regression Model...")
lrModel = LinearRegression().fit(xTrainR, yTrainR)
print("Training Random Forest Model...")
rfrModel = RandomForestRegressor(n_estimators=300, random_state=42, max_depth=25).fit(xTrainR, yTrainR)
print("Training XGB Model...")
xgbModel = xgb.XGBRegressor(n_estimators=150, learning_rate=0.1, max_depth=7).fit(xTrainR, yTrainR)

Training Linear Regression Model...
Training Random Forest Model...
Training XGB Model...


In [14]:
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
lrYhat = lrModel.predict(xTestR)
rfrYhat = rfrModel.predict(xTestR)
xgbYhat = xgbModel.predict(xTestR)

lrRMSE = root_mean_squared_error(yTestR, lrYhat)
rfrRMSE = root_mean_squared_error(yTestR, rfrYhat)
xgbRMSE = root_mean_squared_error(yTestR, xgbYhat)

lrMAE = mean_absolute_error(yTestR, lrYhat)
rfrMAE = mean_absolute_error(yTestR, rfrYhat)
xgbMAE = mean_absolute_error(yTestR, xgbYhat)

In [15]:
print(f'Logistic Regression - RMSE: {lrRMSE:.3f}, MAE: {lrMAE:.3f}')
print(f'Random Forest - RMSE: {rfrRMSE:.3f}, MAE: {rfrMAE:.3f}')
print(f'XGB Regressor - RMSE: {xgbRMSE:.3f}, MAE: {xgbMAE:.3f}')

Logistic Regression - RMSE: 7.190, MAE: 5.647
Random Forest - RMSE: 6.539, MAE: 5.203
XGB Regressor - RMSE: 5.968, MAE: 4.739
